# GuidaPlate — LSTM Dietary Pattern Analyzer (v2)
## Improved pipeline: sequence-aware labels + occasion feature + Masking

**Notebook 05b** — does not modify notebook 05 or production artifacts.

### Input Sequence (v2)
Each patient = 6 meal steps: Day 1 Breakfast → Lunch → Dinner; Day 2 Breakfast → Lunch → Dinner.

Each step = 5 features: [potassium, phosphorus, protein_per_kg, sodium, occasion_encoded]

Input shape: (n_patients, 6, 5)

Labels: `outputs/stats/05_risk_labels_v2.csv`
Cache: `models/lstm_sequences_cache_v2.npz`


In [ ]:
import os
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    accuracy_score, precision_score, recall_score, f1_score, ConfusionMatrixDisplay,
)
import joblib
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Masking, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.utils import to_categorical
try:
    plt.style.use('seaborn-v0_8')
except OSError:
    plt.style.use('seaborn')
%matplotlib inline
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")
RANDOM_STATE = 42
TEST_SIZE = 0.2
ALPHA = 0.05
STAGE_ORDER = ['G2', 'G3a', 'G3b', 'G4']
RISK_CLASSES = ['LOW', 'MODERATE', 'HIGH']
RISK_ENCODE = {c: i for i, c in enumerate(RISK_CLASSES)}
MEAL_OCCASIONS = {
    1: 'D1_Breakfast', 2: 'D1_Lunch', 3: 'D1_Dinner',
    4: 'D2_Breakfast', 5: 'D2_Lunch', 6: 'D2_Dinner',
}
KDOQI = {
    'G2':  {'potassium': 3500, 'phosphorus': 1000, 'protein_per_kg': 0.8, 'sodium': 2300},
    'G3a': {'potassium': 3000, 'phosphorus': 800,  'protein_per_kg': 0.6, 'sodium': 2300},
    'G3b': {'potassium': 3000, 'phosphorus': 800,  'protein_per_kg': 0.6, 'sodium': 2300},
    'G4':  {'potassium': 2500, 'phosphorus': 700,  'protein_per_kg': 0.55, 'sodium': 2300},
}
def project_root():
    p = Path('.').resolve()
    if p.name == 'notebooks':
        return p.parent
    if (p / 'data').exists():
        return p
    return p.parent
ROOT = project_root()
FIG_DIR = ROOT / 'outputs' / 'figures'
STATS_DIR = ROOT / 'outputs' / 'stats'
MODEL_DIR = ROOT / 'models'
FIG_DIR.mkdir(parents=True, exist_ok=True)
STATS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(f"Project root: {ROOT}")


## Section 3 — Load Individual Food Files

NHANES DR1IFF and DR2IFF contain every individual food item each participant ate on Day 1 and Day 2.

Key columns used:
- SEQN: participant ID
- DR1ILINE/DR2ILINE: food line number
- DR1DRSTZ: dietary recall status
- DR1_020/DR2_020: meal occasion code (1=Breakfast, 2=Lunch, 3=Dinner, 4=Snack, 5=Drink, 6=Infant)
- DR1IPOTA/DR2IPOTA: potassium (mg)
- DR1IPHOS/DR2IPHOS: phosphorus (mg)
- DR1IPROT/DR2IPROT: protein (g)
- DR1ISODI/DR2ISODI: sodium (mg)

In [ ]:
def load_iff(path: Path) -> pd.DataFrame:
    try:
        import pyreadstat
        df, _ = pyreadstat.read_xpt(str(path))
        print(f"Loaded {path.name} via pyreadstat")
        return df
    except Exception as exc:
        print(f"pyreadstat unavailable ({exc}); using pandas.read_sas")
        return pd.read_sas(path, format='xport')


def pick_col(df: pd.DataFrame, candidates: list[str]) -> str:
    for c in candidates:
        if c in df.columns:
            return c
    for c in candidates:
        alt = c.replace('.', '_')
        if alt in df.columns:
            return alt
    matches = [col for col in df.columns if any(c.replace('.', '_') in str(col) for c in candidates)]
    if matches:
        return matches[0]
    raise KeyError(f"None of {candidates} found. Available sample: {list(df.columns)[:20]}")


def standardize_iff(df: pd.DataFrame, day: int) -> pd.DataFrame:
    prefix = f'DR{day}'
    rename = {
        pick_col(df, [f'{prefix}.020', f'{prefix}_020']): 'meal_code',
        pick_col(df, [f'{prefix}IPOTA']): 'potassium',
        pick_col(df, [f'{prefix}IPHOS']): 'phosphorus',
        pick_col(df, [f'{prefix}IPROT']): 'protein',
        pick_col(df, [f'{prefix}ISODI']): 'sodium',
    }
    out = df.rename(columns=rename)
    return out[['SEQN', 'meal_code', 'potassium', 'phosphorus', 'protein', 'sodium']].copy()


print("Loading DR1IFF...")
iff1_path = ROOT / 'data' / 'raw' / 'nhanes' / 'DR1IFF_J.xpt'
iff1 = load_iff(iff1_path)
print(f"DR1IFF shape: {iff1.shape}")

print("Loading DR2IFF...")
iff2_path = ROOT / 'data' / 'raw' / 'nhanes' / 'DR2IFF_J.xpt'
iff2 = load_iff(iff2_path)
print(f"DR2IFF shape: {iff2.shape}")

print("Loading CKD cohort...")
cohort = pd.read_csv(ROOT / 'data' / 'processed' / 'ckd_cohort_final.csv')
print(f"CKD cohort: {cohort.shape}")

labels = pd.read_csv(ROOT / 'outputs' / 'stats' / '05_risk_labels_v2.csv')
print(f"Risk labels: {labels.shape}")

cohort = cohort.merge(labels[['SEQN', 'risk_label']], on='SEQN', how='inner')
print(f"Cohort with labels: {cohort.shape}")
print()
print("CKD patients in cohort:")
print(cohort['ckd_stage'].value_counts().reindex(STAGE_ORDER))


## Section 4 — Build Meal Sequences (v2)

Fix 2: append deterministic `occasion_encoded` as 5th feature per slot.
Slots 0/3 → 0.00 (Breakfast), 1/4 → 0.33 (Lunch), 2/5 → 0.67 (Dinner).


In [ ]:
ckd_seqns = set(cohort['SEQN'])
print(f"CKD patients to process: {len(ckd_seqns)}")

OCCASION_BY_SLOT = {0: 0.00, 1: 0.33, 2: 0.67, 3: 0.00, 4: 0.33, 5: 0.67}

import datetime

def map_meal_slot(day, meal_code):
    if pd.isna(meal_code):
        return 2 if day == 1 else 5
    if isinstance(meal_code, datetime.time):
        code = meal_code.hour * 3600 + meal_code.minute * 60 + meal_code.second
    else:
        try:
            code = float(meal_code)
        except (TypeError, ValueError):
            return 2 if day == 1 else 5
    if code <= 10:
        if code == 1:
            return 0 if day == 1 else 3
        if code == 2:
            return 1 if day == 1 else 4
        return 2 if day == 1 else 5
    if code < 39600:
        return 0 if day == 1 else 3
    if code < 61200:
        return 1 if day == 1 else 4
    return 2 if day == 1 else 5

print("Processing Day 1 foods...")
iff1_ckd = standardize_iff(iff1[iff1['SEQN'].isin(ckd_seqns)].copy(), day=1)
iff1_ckd['meal_slot'] = iff1_ckd['meal_code'].apply(lambda x: map_meal_slot(1, x))
print(f"Day 1 food records for CKD patients: {len(iff1_ckd)}")

print("Processing Day 2 foods...")
iff2_ckd = standardize_iff(iff2[iff2['SEQN'].isin(ckd_seqns)].copy(), day=2)
iff2_ckd['meal_slot'] = iff2_ckd['meal_code'].apply(lambda x: map_meal_slot(2, x))
print(f"Day 2 food records for CKD patients: {len(iff2_ckd)}")

all_foods = pd.concat([
    iff1_ckd[['SEQN', 'meal_slot', 'potassium', 'phosphorus', 'protein', 'sodium']],
    iff2_ckd[['SEQN', 'meal_slot', 'potassium', 'phosphorus', 'protein', 'sodium']],
], ignore_index=True)

for col in ['potassium', 'phosphorus', 'protein', 'sodium']:
    all_foods[col] = pd.to_numeric(all_foods[col], errors='coerce').fillna(0)

meal_nutrients = all_foods.groupby(['SEQN', 'meal_slot'])[['potassium', 'phosphorus', 'protein', 'sodium']].sum().reset_index()
print(f"Meal nutrient records: {len(meal_nutrients)}")

sequence_data = []
sequence_labels = []
sequence_seqns = []
cohort_weights = cohort[['SEQN', 'weight_kg', 'ckd_stage', 'risk_label']].copy()
n_processed = 0
n_skipped = 0

for _, patient in cohort_weights.iterrows():
    seqn = patient['SEQN']
    weight = patient['weight_kg']
    risk = patient['risk_label']
    if pd.isna(weight) or weight <= 0:
        n_skipped += 1
        continue
    if risk not in RISK_CLASSES:
        n_skipped += 1
        continue

    seq = np.zeros((6, 5))
    for slot in range(6):
        seq[slot, 4] = OCCASION_BY_SLOT[slot]

    patient_meals = meal_nutrients[meal_nutrients['SEQN'] == seqn]
    for _, meal in patient_meals.iterrows():
        slot = int(meal['meal_slot'])
        if 0 <= slot <= 5:
            seq[slot, 0] = meal['potassium']
            seq[slot, 1] = meal['phosphorus']
            seq[slot, 2] = meal['protein'] / weight
            seq[slot, 3] = meal['sodium']
            seq[slot, 4] = OCCASION_BY_SLOT[slot]

    sequence_data.append(seq)
    sequence_labels.append(risk)
    sequence_seqns.append(seqn)
    n_processed += 1

X_seq = np.array(sequence_data)
y_seq = np.array(sequence_labels)

print(f"Sequences built: {n_processed}")
print(f"Skipped: {n_skipped}")
print(f"Sequence array shape: {X_seq.shape}")
assert X_seq.shape[1:] == (6, 5), f"Expected (6, 5), got {X_seq.shape[1:]}"

print("Risk label distribution:")
unique, counts = np.unique(y_seq, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  {u}: {c} ({c/len(y_seq)*100:.1f}%)")

np.savez(
    MODEL_DIR / 'lstm_sequences_cache_v2.npz',
    sequences=X_seq,
    labels=y_seq,
    patient_ids=np.array(sequence_seqns),
)
print(f"Cached {len(X_seq)} sequences to models/lstm_sequences_cache_v2.npz")



## Section 5 — Normalize and Encode

## Section 5b — Train/Test Split + Truncated Augmentation (v2)

Fix 3: train on **raw** sequences with `Masking(mask_value=0.0)`.
Padding = all four nutrient features are 0 (occasion alone 0 does not mask).
Truncated augmentation zero-pads all 5 features in future slots.


In [ ]:
y_encoded = np.array([RISK_ENCODE[r] for r in y_seq])

print("Label encoding:")
for i, cls in enumerate(RISK_CLASSES):
    print(f"  {i} = {cls}")

n_patients, n_steps, n_features = X_seq.shape
print(f"Sequence shape (raw): {X_seq.shape}")

# Scaler for artifact export only (nutrients); model trains on raw values with Masking
scaler = StandardScaler()
nutrient_mask = X_seq.reshape(-1, n_features)[:, :4].any(axis=1)
scaler.fit(X_seq.reshape(-1, n_features)[nutrient_mask][:, :4])

X_train_raw, X_test_raw, y_train_enc, y_test_enc = train_test_split(
    X_seq, y_encoded,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_encoded,
)

print(f"Training sequences (raw): {len(X_train_raw)}")
print(f"Test sequences (raw): {len(X_test_raw)}")


def augment_with_truncated_sequences(X, y, n_steps=6):
    X_augmented = [X.copy()]
    y_augmented = [y.copy()]
    for cutoff in range(1, n_steps):
        X_truncated = X.copy()
        X_truncated[:, cutoff:, :] = 0
        X_augmented.append(X_truncated)
        y_augmented.append(y.copy())
    return np.concatenate(X_augmented, axis=0), np.concatenate(y_augmented, axis=0)


X_train_aug_raw, y_train_aug = augment_with_truncated_sequences(
    X_train_raw, y_train_enc, n_steps=n_steps
)
print(f"Augmented training sequences (raw): {X_train_aug_raw.shape}")

# Class weights for imbalanced labels
classes = np.unique(y_train_aug)
class_weights_arr = compute_class_weight('balanced', classes=classes, y=y_train_aug)
class_weight = {int(c): float(w) for c, w in zip(classes, class_weights_arr)}
print("Class weights:", class_weight)

X_train = X_train_aug_raw
X_test = X_test_raw
y_train = y_train_aug
y_test = y_test_enc



## Section 6 — LSTM v2 Architecture (Masking)

| Layer | Type | Units |
|---|---|---|
| 1 | Masking | mask_value=0.0 |
| 2 | LSTM | 64 |
| 3 | Dropout | 0.3 |
| 4 | LSTM | 32 |
| 5 | Dropout | 0.3 |
| 6 | Dense | 3 (softmax) |


## Section 6b — Skipped Hyperparameter Tuning

v2 uses the fixed architecture from the improvement spec (Masking + 64/32 LSTM).


In [ ]:
print("Skipping hyperparameter tuning for v2 — using fixed Masking architecture.")


## Section 7 — Train Best LSTM Configuration


In [ ]:
print("Training LSTM v2 with Masking layer")
print("=" * 45)

tf.random.set_seed(RANDOM_STATE)

model = Sequential([
    Masking(mask_value=0.0, input_shape=(n_steps, n_features)),
    LSTM(64, return_sequences=True),
    Dropout(0.3),
    LSTM(32),
    Dropout(0.3),
    Dense(3, activation='softmax'),
], name='GuidaPlate_LSTM_v2')

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

model.summary()
print(f"Total parameters: {model.count_params():,}")

callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    ModelCheckpoint(filepath=str(MODEL_DIR / 'lstm_v2_best.keras'), monitor='val_loss', save_best_only=True, verbose=0),
]

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=callbacks,
    class_weight=class_weight,
    verbose=1,
)

best_epoch = int(np.argmin(history.history['val_loss']) + 1)
print("Training complete.")
print(f"Best epoch: {best_epoch}")



## Section 7b — v2 Evaluation Report

Full metrics on held-out test set vs original LSTM benchmarks.


In [ ]:
print("v2 evaluation in Section 8 below.")


## Section 8 — Performance Metrics

In [ ]:
from sklearn.metrics import classification_report

y_pred_prob = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_prob, axis=1)

v2_accuracy = accuracy_score(y_test, y_pred)
v2_f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
v2_auc = roc_auc_score(y_test, y_pred_prob, multi_class='ovr', average='macro')

cm = confusion_matrix(y_test, y_pred, labels=[0, 1, 2])
print("Confusion matrix:")
print(pd.DataFrame(cm, index=RISK_CLASSES, columns=RISK_CLASSES))
print()
print(classification_report(y_test, y_pred, labels=[0, 1, 2], target_names=RISK_CLASSES, zero_division=0))
print(f"ROC-AUC (macro, ovr): {v2_auc:.4f}")

high_idx = RISK_ENCODE['HIGH']
mod_idx = RISK_ENCODE['MODERATE']
high_tp = ((y_test == high_idx) & (y_pred == high_idx)).sum()
high_fn = ((y_test == high_idx) & (y_pred != high_idx)).sum()
v2_high_sensitivity = high_tp / (high_tp + high_fn) if (high_tp + high_fn) > 0 else 0.0

mod_tp = ((y_test == mod_idx) & (y_pred == mod_idx)).sum()
mod_fn = ((y_test == mod_idx) & (y_pred != mod_idx)).sum()
v2_mod_sensitivity = mod_tp / (mod_tp + mod_fn) if (mod_tp + mod_fn) > 0 else 0.0

print(f"HIGH sensitivity: {v2_high_sensitivity:.4f} ({v2_high_sensitivity*100:.2f}%)")
print(f"MODERATE sensitivity: {v2_mod_sensitivity:.4f} ({v2_mod_sensitivity*100:.2f}%)")

ORIGINAL = {
    'accuracy': 0.9071,
    'f1': 0.9052,
    'auc': 0.9825,
    'high_sensitivity': 0.9360,
    'mod_sensitivity': None,
}

print()
print("=" * 55)
print("METRIC COMPARISON — Original vs V2")
print("=" * 55)
print(f"{'Metric':<18} | {'Original':>10} | {'V2':>10}")
print("-" * 45)
print(f"{'Accuracy':<18} | {ORIGINAL['accuracy']*100:>9.2f}% | {v2_accuracy*100:>9.2f}%")
print(f"{'F1 (weighted)':<18} | {ORIGINAL['f1']:>10.4f} | {v2_f1:>10.4f}")
print(f"{'AUC':<18} | {ORIGINAL['auc']:>10.4f} | {v2_auc:>10.4f}")
print(f"{'HIGH sensitivity':<18} | {ORIGINAL['high_sensitivity']*100:>9.2f}% | {v2_high_sensitivity*100:>9.2f}%")
orig_mod = ORIGINAL['mod_sensitivity']
orig_mod_str = f"{orig_mod*100:.2f}%" if orig_mod is not None else "N/A"
print(f"{'MOD sensitivity':<18} | {orig_mod_str:>10} | {v2_mod_sensitivity*100:>9.2f}%")



## Section 9 — Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'], label='Train', color='#2563EB')
axes[0].plot(history.history['val_accuracy'], label='Validation', color='#DC2626')
axes[0].set_title('LSTM Training Accuracy', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['loss'], label='Train', color='#2563EB')
axes[1].plot(history.history['val_loss'], label='Validation', color='#DC2626')
axes[1].set_title('LSTM Training Loss', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('GuidaPlate LSTM Training History', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / '13_lstm_v2_training_history.png', dpi=150, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred, labels=[0, 1, 2])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=RISK_CLASSES)
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('LSTM Confusion Matrix - GuidaPlate Pattern Analyzer', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / '14_lstm_v2_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

meal_labels = ['D1 Breakfast', 'D1 Lunch', 'D1 Dinner', 'D2 Breakfast', 'D2 Lunch', 'D2 Dinner']
nutrient_names = ['Potassium (mg)', 'Phosphorus (mg)', 'Protein (g/kg)', 'Sodium (mg)']
risk_colors = {'LOW': '#16A34A', 'MODERATE': '#D97706', 'HIGH': '#DC2626'}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for feat_idx, (ax, nutrient) in enumerate(zip(axes, nutrient_names)):
    for risk in RISK_CLASSES:
        risk_idx = RISK_ENCODE[risk]
        mask = y_test == risk_idx
        if mask.sum() == 0:
            continue
        mean_seq = X_test[mask, :, feat_idx].mean(axis=0)
        ax.plot(range(6), mean_seq, label=risk, color=risk_colors[risk], linewidth=2, marker='o')
    ax.set_xticks(range(6))
    ax.set_xticklabels(meal_labels, fontsize=8)
    ax.set_title(f'Average {nutrient} by Risk Level', fontweight='bold')
    ax.set_xlabel('Meal Occasion')
    ax.set_ylabel(nutrient)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Nutrient Patterns Across 6 Meal Occasions - GuidaPlate LSTM Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / '15_lstm_v2_meal_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Saved: {FIG_DIR / '13_lstm_v2_training_history.png'}")
print(f"Saved: {FIG_DIR / '14_lstm_v2_confusion_matrix.png'}")
print(f"Saved: {FIG_DIR / '15_lstm_v2_meal_patterns.png'}")


## Section 10 — Save v2 Artifacts (if metrics >= original)


In [ ]:
label_encoder = {'classes': RISK_CLASSES, 'encode': RISK_ENCODE}

v2_better = (
    v2_accuracy >= ORIGINAL['accuracy'] and
    v2_f1 >= ORIGINAL['f1'] and
    v2_auc >= ORIGINAL['auc'] and
    v2_high_sensitivity >= ORIGINAL['high_sensitivity']
)

if v2_better:
    model.save(MODEL_DIR / 'lstm_v2.keras')
    joblib.dump(scaler, MODEL_DIR / 'lstm_v2_scaler.pkl')
    joblib.dump(label_encoder, MODEL_DIR / 'lstm_v2_label_encoder.pkl')
    print("✅ V2 is better — artifacts saved")
    print("Next step: update backend/config.py to point to v2 artifacts")
else:
    print("❌ V2 did not improve on original")
    print("Keeping original model in production")
    print("Document v2 attempt in Chapter 5")

metrics_df = pd.DataFrame([{
    'model': 'LSTM v2',
    'accuracy': round(v2_accuracy, 4),
    'f1_score': round(v2_f1, 4),
    'auc_roc': round(v2_auc, 4),
    'high_risk_sensitivity': round(v2_high_sensitivity, 4),
    'moderate_sensitivity': round(v2_mod_sensitivity, 4),
    'sequence_length': n_steps,
    'features_per_step': n_features,
}])
metrics_df.to_csv(STATS_DIR / '07_lstm_v2_metrics.csv', index=False)
print(f"Metrics saved: {STATS_DIR / '07_lstm_v2_metrics.csv'}")
print("Original models/lstm_final.keras untouched.")



## Ablation Study

Isolates each v2 improvement against the original baseline (90.71% accuracy).

| Experiment | Input | Labels | Change |
|---|---|---|---|
| A — Masking only | (n, 6, 4) original cache | original | `Masking(mask_value=0.0)` |
| B — Occasion feature only | (n, 6, 5) + occasion column | original | 5 features, no masking |
| C — Sequence labels only | (n, 6, 4) original cache | v2 labels | new labels only |
| All-3 (v2 combined) | (n, 6, 5) + masking | v2 labels | from prior v2 run |


In [ ]:
import json
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.layers import Masking

ORIGINAL = {
    'accuracy': 0.9071,
    'f1': 0.9052,
    'auc': 0.9825,
    'high_sensitivity': 0.9360,
}

ALL3 = {
    'accuracy': 0.7213,
    'f1': 0.7494,
    'auc': 0.8499,
    'high_sensitivity': 0.8000,
    'mod_sensitivity': 0.4359,
}

OCCASION_BY_SLOT = {0: 0.00, 1: 0.33, 2: 0.67, 3: 0.00, 4: 0.33, 5: 0.67}


def evaluate_metrics(y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    auc = roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro')
    high_idx = RISK_ENCODE['HIGH']
    mod_idx = RISK_ENCODE['MODERATE']
    high_tp = ((y_true == high_idx) & (y_pred == high_idx)).sum()
    high_fn = ((y_true == high_idx) & (y_pred != high_idx)).sum()
    high_sens = high_tp / (high_tp + high_fn) if (high_tp + high_fn) > 0 else 0.0
    mod_tp = ((y_true == mod_idx) & (y_pred == mod_idx)).sum()
    mod_fn = ((y_true == mod_idx) & (y_pred != mod_idx)).sum()
    mod_sens = mod_tp / (mod_tp + mod_fn) if (mod_tp + mod_fn) > 0 else 0.0
    return {
        'accuracy': acc,
        'f1': f1,
        'auc': auc,
        'high_sensitivity': high_sens,
        'mod_sensitivity': mod_sens,
    }


def print_ablation_metrics(label, metrics):
    print(f"\n{'='*55}")
    print(label)
    print('='*55)
    print(f"Accuracy:        {metrics['accuracy']*100:.2f}%")
    print(f"F1 (weighted):   {metrics['f1']:.4f}")
    print(f"AUC (macro):     {metrics['auc']:.4f}")
    print(f"HIGH sensitivity:{metrics['high_sensitivity']*100:.2f}%")
    print(f"MOD sensitivity: {metrics['mod_sensitivity']*100:.2f}%")
    print(classification_report(
        metrics['_y_test'], metrics['_y_pred'],
        labels=[0, 1, 2], target_names=RISK_CLASSES, zero_division=0,
    ))


def ablation_decision(name, metrics):
    ok = (
        metrics['accuracy'] >= ORIGINAL['accuracy'] and
        metrics['f1'] >= ORIGINAL['f1'] and
        metrics['auc'] >= ORIGINAL['auc'] and
        metrics['high_sensitivity'] >= ORIGINAL['high_sensitivity']
    )
    if ok:
        print(f"✅ {name} improves or matches original")
    else:
        print(f"❌ {name} does not improve original")
        drops = []
        if metrics['accuracy'] < ORIGINAL['accuracy']:
            drops.append(f"accuracy {metrics['accuracy']*100:.2f}% vs {ORIGINAL['accuracy']*100:.2f}% ({(metrics['accuracy']-ORIGINAL['accuracy'])*100:+.2f}pp)")
        if metrics['f1'] < ORIGINAL['f1']:
            drops.append(f"F1 {metrics['f1']:.4f} vs {ORIGINAL['f1']:.4f} ({metrics['f1']-ORIGINAL['f1']:+.4f})")
        if metrics['auc'] < ORIGINAL['auc']:
            drops.append(f"AUC {metrics['auc']:.4f} vs {ORIGINAL['auc']:.4f} ({metrics['auc']-ORIGINAL['auc']:+.4f})")
        if metrics['high_sensitivity'] < ORIGINAL['high_sensitivity']:
            drops.append(
                f"HIGH sens {metrics['high_sensitivity']*100:.2f}% vs "
                f"{ORIGINAL['high_sensitivity']*100:.2f}% "
                f"({(metrics['high_sensitivity']-ORIGINAL['high_sensitivity'])*100:+.2f}pp)"
            )
        for d in drops:
            print(f"   ↓ {d}")


def train_ablation(X, y_labels, model, save_name, seed=42):
    y_encoded = np.array([RISK_ENCODE[r] for r in y_labels])
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=y_encoded,
    )
    classes = np.unique(y_train)
    cw = compute_class_weight('balanced', classes=classes, y=y_train)
    class_weight = {int(c): float(w) for c, w in zip(classes, cw)}

    tf.random.set_seed(seed)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    early_stop = EarlyStopping(
        monitor='val_loss', patience=10, restore_best_weights=True, verbose=0,
    )
    model.fit(
        X_train, y_train,
        epochs=50,
        batch_size=32,
        validation_split=0.2,
        callbacks=[early_stop],
        class_weight=class_weight,
        verbose=0,
    )
    model.save(MODEL_DIR / save_name)

    y_prob = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    m = evaluate_metrics(y_test, y_pred, y_prob)
    m['_y_test'] = y_test
    m['_y_pred'] = y_pred
    return m


# Load original cache + labels
cache_orig = np.load(MODEL_DIR / 'lstm_sequences_cache.npz', allow_pickle=True)
X_orig = cache_orig['sequences']
patient_ids = cache_orig['patient_ids']

labels_orig_df = pd.read_csv(STATS_DIR / '05_risk_labels.csv')
label_map_orig = dict(zip(labels_orig_df['SEQN'], labels_orig_df['risk_label']))
y_orig = np.array([label_map_orig[int(s)] for s in patient_ids])

labels_v2_df = pd.read_csv(STATS_DIR / '05_risk_labels_v2.csv')
label_map_v2 = dict(zip(labels_v2_df['SEQN'], labels_v2_df['risk_label']))
y_v2 = np.array([label_map_v2[int(s)] for s in patient_ids])

print(f"Original cache shape: {X_orig.shape}")

# ── Ablation A — Masking only ─────────────────────────────────────────────
model_a = Sequential([
    Masking(mask_value=0.0, input_shape=(6, 4)),
    LSTM(64, return_sequences=True),
    Dropout(0.3),
    LSTM(32),
    Dropout(0.3),
    Dense(3, activation='softmax'),
])
metrics_a = train_ablation(X_orig, y_orig, model_a, 'lstm_ablation_a.keras')
print_ablation_metrics('A — Masking only', metrics_a)
ablation_decision('A', metrics_a)

# ── Ablation B — Occasion feature only ────────────────────────────────────
X_occ = np.zeros((len(X_orig), 6, 5))
X_occ[:, :, :4] = X_orig
for slot, val in OCCASION_BY_SLOT.items():
    X_occ[:, slot, 4] = val

model_b = Sequential([
    LSTM(64, return_sequences=True, input_shape=(6, 5)),
    Dropout(0.3),
    LSTM(32),
    Dropout(0.3),
    Dense(3, activation='softmax'),
])
metrics_b = train_ablation(X_occ, y_orig, model_b, 'lstm_ablation_b.keras')
print_ablation_metrics('B — Occasion feature only', metrics_b)
ablation_decision('B', metrics_b)

# ── Ablation C — Sequence-aware labels only ───────────────────────────────
model_c = Sequential([
    LSTM(64, return_sequences=True, input_shape=(6, 4)),
    Dropout(0.3),
    LSTM(32),
    Dropout(0.3),
    Dense(3, activation='softmax'),
])
metrics_c = train_ablation(X_orig, y_v2, model_c, 'lstm_ablation_c.keras')
print_ablation_metrics('C — Sequence labels only', metrics_c)
ablation_decision('C', metrics_c)

# ── Final comparison table ───────────────────────────────────────────────
rows = [
    {'experiment': 'Original', **{k: ORIGINAL[k] for k in ['accuracy', 'f1', 'auc', 'high_sensitivity']}, 'mod_sensitivity': np.nan},
    {'experiment': 'A-Mask', **{k: metrics_a[k] for k in ['accuracy', 'f1', 'auc', 'high_sensitivity', 'mod_sensitivity']}},
    {'experiment': 'B-Occ', **{k: metrics_b[k] for k in ['accuracy', 'f1', 'auc', 'high_sensitivity', 'mod_sensitivity']}},
    {'experiment': 'C-Label', **{k: metrics_c[k] for k in ['accuracy', 'f1', 'auc', 'high_sensitivity', 'mod_sensitivity']}},
    {'experiment': 'All-3', **{k: ALL3[k] for k in ['accuracy', 'f1', 'auc', 'high_sensitivity', 'mod_sensitivity']}},
]
ablation_df = pd.DataFrame(rows)

print('\n' + '=' * 72)
print('ABLATION COMPARISON TABLE')
print('=' * 72)
print(f"{'Metric':<14} | {'Original':>9} | {'A-Mask':>7} | {'B-Occ':>7} | {'C-Label':>7} | {'All-3':>7}")
print('-' * 72)
print(
    f"{'Accuracy':<14} | {ORIGINAL['accuracy']*100:>8.2f}% | "
    f"{metrics_a['accuracy']*100:>6.2f}% | {metrics_b['accuracy']*100:>6.2f}% | "
    f"{metrics_c['accuracy']*100:>6.2f}% | {ALL3['accuracy']*100:>6.2f}%"
)
print(
    f"{'F1 weighted':<14} | {ORIGINAL['f1']:>9.4f} | "
    f"{metrics_a['f1']:>7.4f} | {metrics_b['f1']:>7.4f} | "
    f"{metrics_c['f1']:>7.4f} | {ALL3['f1']:>7.4f}"
)
print(
    f"{'AUC':<14} | {ORIGINAL['auc']:>9.4f} | "
    f"{metrics_a['auc']:>7.4f} | {metrics_b['auc']:>7.4f} | "
    f"{metrics_c['auc']:>7.4f} | {ALL3['auc']:>7.4f}"
)
print(
    f"{'HIGH sens':<14} | {ORIGINAL['high_sensitivity']*100:>8.2f}% | "
    f"{metrics_a['high_sensitivity']*100:>6.2f}% | {metrics_b['high_sensitivity']*100:>6.2f}% | "
    f"{metrics_c['high_sensitivity']*100:>6.2f}% | {ALL3['high_sensitivity']*100:>6.2f}%"
)
print(
    f"{'MOD sens':<14} | {'~0%':>9} | "
    f"{metrics_a['mod_sensitivity']*100:>6.2f}% | {metrics_b['mod_sensitivity']*100:>6.2f}% | "
    f"{metrics_c['mod_sensitivity']*100:>6.2f}% | {ALL3['mod_sensitivity']*100:>6.2f}%"
)

ablation_df.to_csv(STATS_DIR / '08_lstm_ablation_results.csv', index=False)
print(f"\nSaved: {STATS_DIR / '08_lstm_ablation_results.csv'}")
print('Ablation models saved: lstm_ablation_a/b/c.keras')
print('Original models/lstm_final.keras untouched.')


## Ablation Study v2 — Controlled

Reruns A/B/C ablations with **the same preprocessing as notebook 05** (StandardScaler + truncated-sequence augmentation on train only).

### Reference — exact cells from notebook 05

**Scaler + split + augmentation + scale** (notebook 05, Section 5b):

```python
y_encoded = np.array([RISK_ENCODE[r] for r in y_seq])
y_cat = to_categorical(y_encoded, num_classes=3)

n_patients, n_steps, n_features = X_seq.shape
X_flat = X_seq.reshape(-1, n_features)
scaler = StandardScaler()
scaler.fit(X_flat)  # notebook 05 fits on ALL sequences before split

# Split on RAW sequences; scale after augmentation
X_train_raw, X_test_raw, y_train, y_test, idx_train, idx_test = train_test_split(
    X_seq, y_cat, y_encoded,
    test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_encoded,
)

def augment_with_truncated_sequences(X, y, n_steps=6):
    """For each sequence, add truncated+padded versions (cutoffs 1..5), same label."""
    X_augmented = [X.copy()]
    y_augmented = [y.copy()]
    for cutoff in range(1, n_steps):
        X_truncated = X.copy()
        X_truncated[:, cutoff:, :] = 0
        X_augmented.append(X_truncated)
        y_augmented.append(y.copy())
    return np.concatenate(X_augmented, axis=0), np.concatenate(y_augmented, axis=0)

X_train_aug_raw, y_train_aug = augment_with_truncated_sequences(
    X_train_raw, y_train, n_steps=n_steps
)

def scale_sequences(X, scaler):
    n = X.shape[0]
    flat = X.reshape(-1, n_features)
    return scaler.transform(flat).reshape(n, n_steps, n_features)

X_train = scale_sequences(X_train_aug_raw, scaler)
X_test = scale_sequences(X_test_raw, scaler)
y_train = y_train_aug
```

**Note:** Notebook 05 fits `StandardScaler` on the **full** `X_seq` flat array before the train/test split (not train-only). Controlled ablations replicate this exactly.

### A2 / B2 / C2 changes

| Exp | Input | Labels | Model change |
|-----|-------|--------|--------------|
| A2 | (n,6,4) | original | + Masking after explicit zero-out of padded scaled slots |
| B2 | (n,6,5) | original | 5 features, no masking |
| C2 | (n,6,4) | v2 | new labels only |


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.layers import Masking
from tensorflow.keras.utils import to_categorical

ORIGINAL = {
    'accuracy': 0.9071,
    'f1': 0.9052,
    'auc': 0.9825,
    'high_sensitivity': 0.9360,
}

ALL3 = {
    'accuracy': 0.7213,
    'f1': 0.7494,
    'auc': 0.8499,
    'high_sensitivity': 0.8000,
    'mod_sensitivity': 0.4359,
}

OCCASION_BY_SLOT = {0: 0.00, 1: 0.33, 2: 0.67, 3: 0.00, 4: 0.33, 5: 0.67}
N_STEPS = 6


def augment_with_truncated_sequences(X, y, n_steps=6):
    """Exact copy from notebook 05 — truncated+padded versions, same label."""
    X_augmented = [X.copy()]
    y_augmented = [y.copy()]
    for cutoff in range(1, n_steps):
        X_truncated = X.copy()
        X_truncated[:, cutoff:, :] = 0
        X_augmented.append(X_truncated)
        y_augmented.append(y.copy())
    return np.concatenate(X_augmented, axis=0), np.concatenate(y_augmented, axis=0)


def scale_sequences(X, scaler, n_features, n_steps=6):
    """Exact copy from notebook 05."""
    n = X.shape[0]
    flat = X.reshape(-1, n_features)
    return scaler.transform(flat).reshape(n, n_steps, n_features)


def zero_out_scaled_padding(X_scaled, X_raw):
    """A2: after scaling, force padded timesteps back to 0.0 for Masking layer."""
    X_out = X_scaled.copy()
    pad_mask = (X_raw == 0).all(axis=-1)
    for i in range(X_out.shape[0]):
        X_out[i, pad_mask[i], :] = 0.0
    return X_out


def controlled_preprocess(X_seq, y_labels, n_features, apply_mask_zero=False):
    """Notebook-05 preprocessing: scaler on full X_seq, split, augment train, scale."""
    y_encoded = np.array([RISK_ENCODE[r] for r in y_labels])
    y_cat = to_categorical(y_encoded, num_classes=3)

    scaler = StandardScaler()
    scaler.fit(X_seq.reshape(-1, n_features))

    X_train_raw, X_test_raw, y_train_cat, y_test_cat, y_train_enc, y_test_enc = train_test_split(
        X_seq, y_cat, y_encoded,
        test_size=0.2, random_state=RANDOM_STATE, stratify=y_encoded,
    )

    X_train_aug_raw, y_train_aug_cat = augment_with_truncated_sequences(
        X_train_raw, y_train_cat, n_steps=N_STEPS,
    )

    X_train_scaled = scale_sequences(X_train_aug_raw, scaler, n_features)
    X_test_scaled = scale_sequences(X_test_raw, scaler, n_features)

    if apply_mask_zero:
        X_train_scaled = zero_out_scaled_padding(X_train_scaled, X_train_aug_raw)
        X_test_scaled = zero_out_scaled_padding(X_test_scaled, X_test_raw)

    y_train_int = np.argmax(y_train_aug_cat, axis=1)
    classes = np.unique(y_train_int)
    cw = compute_class_weight('balanced', classes=classes, y=y_train_int)
    class_weight = {int(c): float(w) for c, w in zip(classes, cw)}

    return X_train_scaled, X_test_scaled, y_train_int, y_test_enc, class_weight


def evaluate_metrics(y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    auc = roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro')
    high_idx, mod_idx = RISK_ENCODE['HIGH'], RISK_ENCODE['MODERATE']
    high_tp = ((y_true == high_idx) & (y_pred == high_idx)).sum()
    high_fn = ((y_true == high_idx) & (y_pred != high_idx)).sum()
    high_sens = high_tp / (high_tp + high_fn) if (high_tp + high_fn) > 0 else 0.0
    mod_tp = ((y_true == mod_idx) & (y_pred == mod_idx)).sum()
    mod_fn = ((y_true == mod_idx) & (y_pred != mod_idx)).sum()
    mod_sens = mod_tp / (mod_tp + mod_fn) if (mod_tp + mod_fn) > 0 else 0.0
    return {
        'accuracy': acc, 'f1': f1, 'auc': auc,
        'high_sensitivity': high_sens, 'mod_sensitivity': mod_sens,
        '_y_test': y_true, '_y_pred': y_pred,
    }


def build_model(n_features, use_masking=False):
    layers = []
    if use_masking:
        layers.append(Masking(mask_value=0.0, input_shape=(N_STEPS, n_features)))
    else:
        layers.append(LSTM(64, return_sequences=True, input_shape=(N_STEPS, n_features)))
        layers += [Dropout(0.3), LSTM(32), Dropout(0.3), Dense(3, activation='softmax')]
        return Sequential(layers)
    layers += [
        LSTM(64, return_sequences=True), Dropout(0.3),
        LSTM(32), Dropout(0.3), Dense(3, activation='softmax'),
    ]
    return Sequential(layers)


def train_ablation_controlled(X_train, X_test, y_train, y_test, model, save_name):
    tf.random.set_seed(RANDOM_STATE)
    classes = np.unique(y_train)
    cw = compute_class_weight('balanced', classes=classes, y=y_train)
    class_weight = {int(c): float(w) for c, w in zip(classes, cw)}

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    early_stop = EarlyStopping(
        monitor='val_loss', patience=10, restore_best_weights=True, verbose=0,
    )
    model.fit(
        X_train, y_train, epochs=50, batch_size=32, validation_split=0.2,
        callbacks=[early_stop], class_weight=class_weight, verbose=0,
    )
    model.save(MODEL_DIR / save_name)
    y_prob = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    return evaluate_metrics(y_test, y_pred, y_prob)


def ablation_decision_controlled(name, metrics):
    ok = (
        metrics['accuracy'] >= ORIGINAL['accuracy'] and
        metrics['f1'] >= ORIGINAL['f1'] and
        metrics['auc'] >= ORIGINAL['auc'] and
        metrics['high_sensitivity'] >= ORIGINAL['high_sensitivity']
    )
    tradeoff = (
        metrics['mod_sensitivity'] > 0.20 and
        (ORIGINAL['accuracy'] - metrics['accuracy']) < 0.05
    )
    if ok:
        print(f"✅ {name} improves or matches original — production candidate")
        return 'PASS'
    print(f"❌ {name} does not improve original")
    if metrics['accuracy'] < ORIGINAL['accuracy']:
        print(f"   ↓ accuracy {metrics['accuracy']*100:.2f}% vs {ORIGINAL['accuracy']*100:.2f}% ({(metrics['accuracy']-ORIGINAL['accuracy'])*100:+.2f}pp)")
    if metrics['f1'] < ORIGINAL['f1']:
        print(f"   ↓ F1 {metrics['f1']:.4f} vs {ORIGINAL['f1']:.4f} ({metrics['f1']-ORIGINAL['f1']:+.4f})")
    if metrics['auc'] < ORIGINAL['auc']:
        print(f"   ↓ AUC {metrics['auc']:.4f} vs {ORIGINAL['auc']:.4f} ({metrics['auc']-ORIGINAL['auc']:+.4f})")
    if metrics['high_sensitivity'] < ORIGINAL['high_sensitivity']:
        print(f"   ↓ HIGH sens {metrics['high_sensitivity']*100:.2f}% vs {ORIGINAL['high_sensitivity']*100:.2f}% ({(metrics['high_sensitivity']-ORIGINAL['high_sensitivity'])*100:+.2f}pp)")
    if tradeoff:
        print(f"⚠ TRADE-OFF — review manually: MOD sens {metrics['mod_sensitivity']*100:.1f}% with accuracy drop {(ORIGINAL['accuracy']-metrics['accuracy'])*100:.1f}pp")
        return 'TRADE-OFF'
    return 'FAIL'


def print_metrics(label, m):
    print(f"\n{'='*55}\n{label}\n{'='*55}")
    print(f"Accuracy: {m['accuracy']*100:.2f}%  F1: {m['f1']:.4f}  AUC: {m['auc']:.4f}")
    print(f"HIGH sens: {m['high_sensitivity']*100:.2f}%  MOD sens: {m['mod_sensitivity']*100:.2f}%")
    print(classification_report(m['_y_test'], m['_y_pred'], labels=[0,1,2], target_names=RISK_CLASSES, zero_division=0))


# ── Load data ─────────────────────────────────────────────────────────────
cache = np.load(MODEL_DIR / 'lstm_sequences_cache.npz', allow_pickle=True)
X_orig = cache['sequences']
patient_ids = cache['patient_ids']

labels_orig = pd.read_csv(STATS_DIR / '05_risk_labels.csv')
labels_v2 = pd.read_csv(STATS_DIR / '05_risk_labels_v2.csv')
map_orig = dict(zip(labels_orig['SEQN'], labels_orig['risk_label']))
map_v2 = dict(zip(labels_v2['SEQN'], labels_v2['risk_label']))
y_orig = np.array([map_orig[int(s)] for s in patient_ids])
y_v2 = np.array([map_v2[int(s)] for s in patient_ids])

X_occ = np.zeros((len(X_orig), N_STEPS, 5))
X_occ[:, :, :4] = X_orig
for slot, val in OCCASION_BY_SLOT.items():
    X_occ[:, slot, 4] = val

print(f"Original cache: {X_orig.shape}  |  Occasion cache: {X_occ.shape}")

# ── A2: Masking + controlled preprocessing ────────────────────────────────
Xt, Xe, yt, ye, _ = controlled_preprocess(X_orig, y_orig, n_features=4, apply_mask_zero=True)
m_a2 = train_ablation_controlled(Xt, Xe, yt, ye, build_model(4, use_masking=True), 'lstm_ablation_a2.keras')
print_metrics('A2 — Masking + controlled preprocessing', m_a2)
d_a2 = ablation_decision_controlled('A2', m_a2)

# ── B2: Occasion + controlled preprocessing ───────────────────────────────
Xt, Xe, yt, ye, _ = controlled_preprocess(X_occ, y_orig, n_features=5, apply_mask_zero=False)
m_b2 = train_ablation_controlled(Xt, Xe, yt, ye, build_model(5, use_masking=False), 'lstm_ablation_b2.keras')
print_metrics('B2 — Occasion + controlled preprocessing', m_b2)
d_b2 = ablation_decision_controlled('B2', m_b2)

# ── C2: v2 labels + controlled preprocessing ──────────────────────────────
Xt, Xe, yt, ye, _ = controlled_preprocess(X_orig, y_v2, n_features=4, apply_mask_zero=False)
m_c2 = train_ablation_controlled(Xt, Xe, yt, ye, build_model(4, use_masking=False), 'lstm_ablation_c2.keras')
print_metrics('C2 — Sequence labels + controlled preprocessing', m_c2)
d_c2 = ablation_decision_controlled('C2', m_c2)

# ── Comparison table ────────────────────────────────────────────────────
rows = [
    {'experiment': 'Original', 'accuracy': ORIGINAL['accuracy'], 'f1': ORIGINAL['f1'],
     'auc': ORIGINAL['auc'], 'high_sensitivity': ORIGINAL['high_sensitivity'],
     'mod_sensitivity': None, 'decision': 'baseline'},
    {'experiment': 'A2-Mask', **{k: m_a2[k] for k in ['accuracy','f1','auc','high_sensitivity','mod_sensitivity']}, 'decision': d_a2},
    {'experiment': 'B2-Occ', **{k: m_b2[k] for k in ['accuracy','f1','auc','high_sensitivity','mod_sensitivity']}, 'decision': d_b2},
    {'experiment': 'C2-Label', **{k: m_c2[k] for k in ['accuracy','f1','auc','high_sensitivity','mod_sensitivity']}, 'decision': d_c2},
    {'experiment': 'All-3', **{k: ALL3[k] for k in ['accuracy','f1','auc','high_sensitivity','mod_sensitivity']}, 'decision': 'FAIL'},
]
df_ctrl = pd.DataFrame(rows)

print('\n' + '=' * 72)
print('CONTROLLED ABLATION COMPARISON TABLE')
print('=' * 72)
print(f"{'Metric':<12} | {'Original':>9} | {'A2-Mask':>7} | {'B2-Occ':>7} | {'C2-Label':>8} | {'All-3':>7}")
print('-' * 72)
print(f"{'Accuracy':<12} | {ORIGINAL['accuracy']*100:>8.2f}% | {m_a2['accuracy']*100:>6.2f}% | {m_b2['accuracy']*100:>6.2f}% | {m_c2['accuracy']*100:>7.2f}% | {ALL3['accuracy']*100:>6.2f}%")
print(f"{'F1':<12} | {ORIGINAL['f1']:>9.4f} | {m_a2['f1']:>7.4f} | {m_b2['f1']:>7.4f} | {m_c2['f1']:>8.4f} | {ALL3['f1']:>7.4f}")
print(f"{'AUC':<12} | {ORIGINAL['auc']:>9.4f} | {m_a2['auc']:>7.4f} | {m_b2['auc']:>7.4f} | {m_c2['auc']:>8.4f} | {ALL3['auc']:>7.4f}")
print(f"{'HIGH sens':<12} | {ORIGINAL['high_sensitivity']*100:>8.2f}% | {m_a2['high_sensitivity']*100:>6.2f}% | {m_b2['high_sensitivity']*100:>6.2f}% | {m_c2['high_sensitivity']*100:>7.2f}% | {ALL3['high_sensitivity']*100:>6.2f}%")
print(f"{'MOD sens':<12} | {'~0%':>9} | {m_a2['mod_sensitivity']*100:>6.2f}% | {m_b2['mod_sensitivity']*100:>6.2f}% | {m_c2['mod_sensitivity']*100:>7.2f}% | {ALL3['mod_sensitivity']*100:>6.2f}%")

df_ctrl.to_csv(STATS_DIR / '08_lstm_ablation_controlled.csv', index=False)
print(f"\nSaved: {STATS_DIR / '08_lstm_ablation_controlled.csv'}")
print('models/lstm_final.keras untouched.')
